In [1]:
%pip install "unitycatalog-ai[databricks]"


c:\Users\hp\Documents\GitHub\cs4603\27100380_cs4603-pa4\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


NameError: name 'dbutils' is not defined

In [2]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

CATALOG = "cs4603"
SCHEMA = "default"
PREFIX = "s27100380"

client = DatabricksFunctionClient()
print("Unity Catalog packages loaded successfully.")

Unity Catalog packages loaded successfully.


In [5]:
def growth_rate(
    start_value: float,
    rate: float,
    years: int,
) -> float:
    """Project a starting value using compound annual growth.

    Args:
        start_value: Initial numeric value before growth.
        rate: Annual growth rate expressed as a decimal, such as 0.08 for 8%.
        years: Number of complete years over which growth is compounded.

    Returns:
        The projected value after compound growth.
    """
    return start_value * (1.0 + rate) ** years


def percentage_change(
    old_value: float,
    new_value: float,
) -> float:
    """Calculate the percentage change between two values.

    Args:
        old_value: Original baseline value. It must not be zero.
        new_value: Updated value being compared with the baseline.

    Returns:
        Percentage change, where a positive value means an increase and a
        negative value means a decrease.

    Raises:
        ValueError: If old_value is zero.
    """
    if old_value == 0:
        raise ValueError("old_value must not be zero")

    return ((new_value - old_value) / abs(old_value)) * 100.0


def compare_values(
    first_value: float,
    second_value: float,
) -> str:
    """Compare two numeric values and describe their difference.

    Args:
        first_value: First value to compare.
        second_value: Second value to compare.

    Returns:
        A description identifying the larger value and the absolute difference.
    """
    if first_value == second_value:
        return f"{first_value:g} and {second_value:g} are equal"

    larger = max(first_value, second_value)
    smaller = min(first_value, second_value)
    difference = larger - smaller

    return (
        f"{larger:g} is larger than {smaller:g} "
        f"by an absolute difference of {difference:g}"
    )

In [6]:
python_functions = [
    growth_rate,
    percentage_change,
    compare_values,
]

for function in python_functions:
    created = client.create_python_function(
        func=function,
        catalog=CATALOG,
        schema=SCHEMA,
        replace=True,
    )
    print(f"Registered: {CATALOG}.{SCHEMA}.{function.__name__}")

Registered: cs4603.default.s27100380_growth_rate
Registered: cs4603.default.s27100380_percentage_change
Registered: cs4603.default.s27100380_compare_values


In [8]:
tests = [
    (
        "cs4603.default.growth_rate",
        {
            "start_value": 16.91,
            "rate": 0.08,
            "years": 3,
        },
    ),
    (
        "cs4603.default.percentage_change",
        {
            "old_value": 100.0,
            "new_value": 125.0,
        },
    ),
    (
        "cs4603.default.compare_values",
        {
            "first_value": 16.91,
            "second_value": 18.60,
        },
    ),
]

for function_name, parameters in tests:
    result = client.execute_function(function_name, parameters)
    print(function_name)
    print(result)
    print("-" * 60)

cs4603.default.growth_rate
FunctionExecutionResult(error=None, format='SCALAR', value='21.301729920000003', truncated=None)
------------------------------------------------------------
cs4603.default.percentage_change
FunctionExecutionResult(error=None, format='SCALAR', value='25.0', truncated=None)
------------------------------------------------------------
cs4603.default.compare_values
FunctionExecutionResult(error=None, format='SCALAR', value='18.6 is larger than 16.91 by an absolute difference of 1.69', truncated=None)
------------------------------------------------------------


In [ ]:
%sql
CREATE OR REPLACE FUNCTION cs4603.default.to_billions(
    amount DOUBLE
)
RETURNS DOUBLE
COMMENT 'Convert an amount expressed in base units into billions.'
RETURN amount / 1000000000.0;

In [10]:
SELECT cs4603.default.to_billions(2400000000.0)
       AS amount_in_billions;

SyntaxError: invalid syntax (1301824298.py, line 1)

In [11]:
from uc_tools.register_functions import (
    register_python_functions,
    verify_python_functions,
)

registered_names = register_python_functions()
verification_results = verify_python_functions()

Registered: cs4603.default.growth_rate
Registered: cs4603.default.percentage_change
Registered: cs4603.default.compare_values

Function: cs4603.default.growth_rate
Parameters: {'start_value': 16.91, 'rate': 0.08, 'years': 3}
Result: FunctionExecutionResult(error=None, format='SCALAR', value='21.301729920000003', truncated=None)

Function: cs4603.default.percentage_change
Parameters: {'old_value': 100.0, 'new_value': 125.0}
Result: FunctionExecutionResult(error=None, format='SCALAR', value='25.0', truncated=None)

Function: cs4603.default.compare_values
Parameters: {'first_value': 16.91, 'second_value': 18.6}
Result: FunctionExecutionResult(error=None, format='SCALAR', value='18.6 is larger than 16.91 by an absolute difference of 1.69', truncated=None)
